# FlowGuard -- neural model (PyTorch, LinuxONE)

Trains a network on the features `01_prepare_and_baseline.ipynb` exported, and
scores it through the **same** `flowguard.evaluation.metrics.evaluate()` the
XGBoost experiments use, so the numbers are directly comparable.

**Run `01_prepare_and_baseline.ipynb` first** -- this needs its
`*_features.h5`. Nothing here recomputes features, so iterating on the model
costs minutes rather than a full extraction run.

### Why PyTorch and not Keras

`tensorflow 2.9.3` cannot run on this image. It requires `protobuf < 3.20`;
the image ships `7.35.1`. Setting
`PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python` gets the import through, but
constructing any Keras layer then raises *"RepeatedCompositeFieldContainer
object does not support item assignment"*, and nothing installable fixes it on
s390x. The datathon's own `Fraud_LSTM_Keras_TF.ipynb` and
`Digit_Class_TensorFlow.ipynb` carry no saved outputs, consistent with
TensorFlow never having worked on this image.

`torch 2.1.0a0` -- built from source for s390x -- does work, so the neural
comparison lives here instead.

### Three things that would silently ruin this, handled below
1. **NaNs.** GFP leaves self-transfer rows NaN by design. XGBoost uses that as
   signal; a network propagates it into a NaN loss and dies on epoch 1.
   Imputed here, after scaling.
2. **Scale.** Trees are scale-invariant, networks are not. `StandardScaler` is
   fitted on the **training partition only**, in chunks.
3. **Imbalance.** `pos_weight` in `BCEWithLogitsLoss` is the direct analogue of
   XGBoost's `scale_pos_weight`.


In [ ]:
import os
import sys
import platform

# Must precede any protobuf import. torch pulls in transformers on this image,
# which pulls protobuf, which is too new for the bundled generated code.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

import numpy as np
import torch

torch.set_num_threads(max(1, os.cpu_count() or 1))     # 2 vCPUs here
torch.manual_seed(42)
np.random.seed(42)

print("python  :", sys.version.split()[0])
print("machine :", platform.machine(), "/ byte order:", sys.byteorder)
print("torch   :", torch.__version__)
print("threads :", torch.get_num_threads())
print("device  : cpu (no GPU on this VM)")


## 1. Locate the exported features and the flowguard source

In [ ]:
from pathlib import Path
import h5py

HOME, CWD = Path.home(), Path.cwd()

# ---- Optional overrides (leave None to auto-detect) -----------------------
H5_OVERRIDE = None
FLOWGUARD_SRC_OVERRIDE = None
# ---------------------------------------------------------------------------


def _find(candidates, probe):
    for c in candidates:
        if probe(c):
            return c
    return None


FLOWGUARD_SRC = (
    Path(FLOWGUARD_SRC_OVERRIDE) if FLOWGUARD_SRC_OVERRIDE
    else Path(os.environ["FLOWGUARD_SRC"]) if os.environ.get("FLOWGUARD_SRC")
    else _find([*[p / "src" for p in (CWD, *CWD.parents)],
                *sorted(HOME.glob("*/src")), *sorted(HOME.glob("*/*/src"))],
               lambda c: (c / "flowguard" / "data" / "schema.py").exists())
)
if FLOWGUARD_SRC is None:
    raise FileNotFoundError(
        "Could not find the flowguard 'src' directory. Set "
        "FLOWGUARD_SRC_OVERRIDE above, or export FLOWGUARD_SRC."
    )
sys.path.insert(0, str(FLOWGUARD_SRC))

OUTPUT_DIR = Path(os.environ.get("FLOWGUARD_OUTPUT_DIR", HOME / "flowguard_outputs"))
H5_PATH = (
    Path(H5_OVERRIDE) if H5_OVERRIDE
    else _find(sorted(OUTPUT_DIR.glob("*_features.h5")) + sorted(CWD.glob("data/*_features.h5")),
               lambda c: c.exists())
)
if H5_PATH is None:
    raise FileNotFoundError(
        f"No *_features.h5 found in {OUTPUT_DIR} or {CWD / 'data'}.\n"
        "Run 01_prepare_and_baseline.ipynb first -- it writes that file."
    )

with h5py.File(H5_PATH, "r") as h5:
    N_TRAIN, N_VAL, N_TEST = (int(h5.attrs[k]) for k in ("n_train", "n_val", "n_test"))
    N_FEATURES = int(h5.attrs["n_features"])
    GFP_FEATURE_SOURCE = str(h5.attrs.get("gfp_feature_source", "unknown"))
    FEATURE_NAMES = [n.decode() if isinstance(n, bytes) else str(n)
                     for n in h5["feature_names"][:]]
    y_all = h5["y"][:].astype(int)

TRAIN_SPAN = (0, N_TRAIN)
VAL_SPAN = (N_TRAIN, N_TRAIN + N_VAL)
TEST_SPAN = (N_TRAIN + N_VAL, N_TRAIN + N_VAL + N_TEST)
y_train, y_val, y_test = (y_all[a:b] for a, b in (TRAIN_SPAN, VAL_SPAN, TEST_SPAN))

print("features :", H5_PATH)
print(f"  {N_FEATURES} features, graph source: {GFP_FEATURE_SOURCE}")
print(f"  train {N_TRAIN:,} ({y_train.sum():,} pos)   val {N_VAL:,} ({y_val.sum():,} pos)"
      f"   test {N_TEST:,} ({y_test.sum():,} pos)")
print(f"  test base rate: {y_test.mean():.4%}")


## 2. Scaling

`StandardScaler.partial_fit` over the training partition in chunks, so the
full matrix is never resident. scikit-learn ignores NaNs when computing the
statistics and preserves them through `transform`; they are imputed to 0
afterwards, which -- post-scaling -- means "the training mean".


In [ ]:
from sklearn.preprocessing import StandardScaler

CHUNK = 100_000
scaler = StandardScaler()
with h5py.File(H5_PATH, "r") as h5:
    for start in range(*TRAIN_SPAN, CHUNK):
        stop = min(start + CHUNK, TRAIN_SPAN[1])
        scaler.partial_fit(h5["X"][start:stop])
        print(f"  fitted {stop:>10,} / {N_TRAIN:,}", end="\r", flush=True)

# A zero-variance column would divide by ~0 and produce infinities.
scaler.scale_ = np.where(scaler.scale_ > 0, scaler.scale_, 1.0)
print(f"\nscaler fitted on {N_TRAIN:,} training rows only")


## 3. Batching and the model

Batches are read straight from HDF5, so peak RAM is one batch rather than the
whole partition -- the same discipline `01` uses.

Shuffling is at **batch granularity**: the order of contiguous blocks is
permuted each epoch. Row-level shuffling would mean random-access reads into
HDF5, which is far slower. Rows inside one batch are therefore temporally
adjacent; if training looks unstable, shrink `BATCH_SIZE`.

The last layer emits **logits**, not probabilities, because
`BCEWithLogitsLoss` is numerically stabler than a sigmoid followed by BCE and
is what carries `pos_weight`.


In [ ]:
import random

BATCH_SIZE = 4096


def batches(span, shuffle=False, batch_size=BATCH_SIZE):
    """Yield (X, y) tensors from an HDF5 span: scaled, NaN-free, float32."""
    blocks = [(lo, min(lo + batch_size, span[1])) for lo in range(span[0], span[1], batch_size)]
    if shuffle:
        random.shuffle(blocks)
    with h5py.File(H5_PATH, "r") as h5:
        for lo, hi in blocks:
            X = scaler.transform(h5["X"][lo:hi])
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
            y = h5["y"][lo:hi].astype("float32")
            yield torch.from_numpy(X), torch.from_numpy(y)


model = torch.nn.Sequential(
    torch.nn.Linear(N_FEATURES, 128), torch.nn.ReLU(), torch.nn.Dropout(0.3),
    torch.nn.Linear(128, 64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
    torch.nn.Linear(64, 32), torch.nn.ReLU(),
    torch.nn.Linear(32, 1),                       # logits
)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

# The direct analogue of XGBoost's scale_pos_weight = negatives / positives.
positives = int(y_train.sum())
pos_weight = torch.tensor([(len(y_train) - positives) / max(positives, 1)], dtype=torch.float32)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(model)
print(f"\nparameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"pos_weight : {pos_weight.item():,.0f}")


## 4. Train

Early stopping watches validation PR-AUC, scored with the project's own
`evaluate()` rather than an approximation. The validation partition is the only
thing allowed to influence when training stops; test is never touched here.


In [ ]:
import time
from flowguard.evaluation.metrics import evaluate, per_group_recall

EPOCHS, PATIENCE = 30, 5


def logits_for(span):
    model.eval()
    out = []
    with torch.no_grad():
        for X, _ in batches(span):
            out.append(model(X).squeeze(1).numpy())
    return np.concatenate(out)


history = {"loss": [], "val_pr_auc": []}
best = {"pr_auc": -1.0, "epoch": -1, "state": None}
started = time.perf_counter()

for epoch in range(EPOCHS):
    model.train()
    total, seen = 0.0, 0
    for X, y in batches(TRAIN_SPAN, shuffle=True):
        optimiser.zero_grad()
        loss = loss_fn(model(X).squeeze(1), y)
        loss.backward()
        optimiser.step()
        total += loss.item() * len(y)
        seen += len(y)

    epoch_loss = total / max(seen, 1)
    val_pr_auc = evaluate(y_val, logits_for(VAL_SPAN)).pr_auc
    history["loss"].append(epoch_loss)
    history["val_pr_auc"].append(val_pr_auc)
    print(f"  epoch {epoch + 1:>2}  loss {epoch_loss:.4f}  val PR-AUC {val_pr_auc:.4f}"
          f"  ({time.perf_counter() - started:.0f}s)", flush=True)

    if val_pr_auc > best["pr_auc"]:
        best = {"pr_auc": val_pr_auc, "epoch": epoch,
                "state": {k: v.clone() for k, v in model.state_dict().items()}}
    elif epoch - best["epoch"] >= PATIENCE:
        print(f"  early stop: no val improvement in {PATIENCE} epochs")
        break

model.load_state_dict(best["state"])          # restore the best weights
train_seconds = time.perf_counter() - started
print(f"\ntrained in {train_seconds:.0f}s, best epoch {best['epoch'] + 1} "
      f"(val PR-AUC {best['pr_auc']:.4f})")


## 5. Score it the project's way

Isotonic calibration fitted on validation only -- the same treatment
`flowguard.models.xgb.XGBModel` gives its output, so the scores mean the same
thing -- then scored with the shared `evaluate()`.


In [ ]:
from sklearn.isotonic import IsotonicRegression

raw_val, raw_test = logits_for(VAL_SPAN), logits_for(TEST_SPAN)
calibrator = IsotonicRegression(out_of_bounds="clip").fit(raw_val, y_val)
dnn_val_scores, dnn_scores = calibrator.predict(raw_val), calibrator.predict(raw_test)

dnn_val = evaluate(y_val, dnn_val_scores)
dnn_test = evaluate(y_test, dnn_scores)
print("PyTorch DNN -- test:")
print(dnn_test.summary())


## 6. Compare against the tree models

Picks up `results.json` from `01` if it is there. Gradient-boosted trees are
the strong baseline on tabular data like this, so do not be surprised if the
network loses -- that is a result worth reporting, not a bug to hide.


In [ ]:
import json
import pandas as pd

rows = {"DNN (PyTorch)": dnn_test.to_metadata()}
results_path = OUTPUT_DIR / "results.json"
if results_path.exists():
    prior = json.loads(results_path.read_text(encoding="utf-8"))
    for key, label in (("E0", "E0 (rules)"), ("E1", "E1 (transaction)"),
                       ("E2", f"E2 ({prior.get('E2', {}).get('feature_source', 'graph')})")):
        if key in prior:
            rows[label] = prior[key]["test"]
else:
    print(f"(no {results_path} -- run 01 for the XGBoost comparison)")


def _at(meta, budget, field):
    return next(b[field] for b in meta["budgets"] if b["budget"] == budget)


comparison = pd.DataFrame({
    "PR-AUC": {k: v["pr_auc"] for k, v in rows.items()},
    "ROC-AUC": {k: v["roc_auc"] for k, v in rows.items()},
    "lift": {k: v["lift_over_base_rate"] for k, v in rows.items()},
    "recall@1%": {k: _at(v, 0.01, "recall") for k, v in rows.items()},
    "precision@1%": {k: _at(v, 0.01, "precision") for k, v in rows.items()},
}).sort_values("PR-AUC", ascending=False)
print(comparison.round(4).to_string())


## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(range(1, len(history["loss"]) + 1), history["loss"], marker="o")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("weighted BCE")
axes[0].set_title("Training loss")
axes[1].plot(range(1, len(history["val_pr_auc"]) + 1), history["val_pr_auc"],
             marker="o", color="#C44E52")
axes[1].axvline(best["epoch"] + 1, linestyle="--", color="gray",
                label=f"best epoch ({best['epoch'] + 1})")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation PR-AUC")
axes[1].set_title("Validation PR-AUC"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_training_history.png")
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(7, 6))
precision, recall, _ = precision_recall_curve(y_test, dnn_scores)
ax.plot(recall, precision, color="#937860",
        label=f"DNN (PyTorch)  PR-AUC={dnn_test.pr_auc:.4f}")

e2_path = OUTPUT_DIR / "e2_test_scores.npy"
if e2_path.exists():
    e2_scores = np.load(e2_path)
    if len(e2_scores) == len(y_test):
        p2, r2, _ = precision_recall_curve(y_test, e2_scores)
        label = next((f"E2 (XGBoost)  PR-AUC={v['pr_auc']:.4f}"
                      for k, v in rows.items() if k.startswith("E2")), "E2 (XGBoost)")
        ax.plot(r2, p2, color="#C44E52", label=label)
    else:
        print(f"(e2_test_scores.npy has {len(e2_scores):,} rows for {len(y_test):,} "
              "test rows -- from a different run, so not overlaid)")

ax.axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1,
           label=f"base rate ({y_test.mean():.4%})")
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("Test set -- precision/recall")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_pr_curve.png")
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix

# Colour is the fraction within each true class; the annotation is the raw
# count. A plain 2x2 at this base rate is all true negatives otherwise.
val_threshold = dnn_val.best_f1_threshold
budget_threshold = dnn_test.at_budget(0.01).threshold

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (title, thr) in zip(axes, [
    (f"best-F1 threshold from validation\n(score >= {val_threshold:.4f})", val_threshold),
    (f"1% alert budget\n(top {int(round(len(y_test) * 0.01)):,} scores)", budget_threshold),
]):
    cm = confusion_matrix(y_test, (dnn_scores >= thr).astype(int), labels=[0, 1])
    sns.heatmap(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), annot=cm, fmt=",d",
                cmap="Oranges", cbar=False, ax=ax,
                xticklabels=["predicted clean", "predicted laundering"],
                yticklabels=["actually clean", "actually laundering"])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{title}\nrecall {tp / max(tp + fn, 1):.1%}  |  "
                 f"precision {tp / max(tp + fp, 1):.2%}  |  {fp:,} false alerts", fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_confusion_matrices.png")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(x=dnn_scores[y_test == 0], stat="density", bins=60, color="#4C72B0",
             label="legitimate", alpha=0.6, ax=ax)
sns.histplot(x=dnn_scores[y_test == 1], stat="density", bins=60, color="#C44E52",
             label="laundering", alpha=0.6, ax=ax)
ax.set_xlabel("calibrated DNN score")
ax.set_title("DNN score distribution by class (test set)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dnn_score_distribution.png")
plt.show()


In [ ]:
# Recall per laundering typology @1% budget
with h5py.File(H5_PATH, "r") as h5:
    cats = np.array([c.decode() if isinstance(c, bytes) else str(c)
                     for c in h5["pattern_type_categories"][:]], dtype=object)
    codes = h5["pattern_type_codes"][slice(*TEST_SPAN)]
typologies = np.full(len(codes), "", dtype=object)
seen = codes >= 0
typologies[seen] = cats[codes[seen]]

typology_recall = per_group_recall(y_test, dnn_scores, typologies, budget=0.01)
if typology_recall:
    typ = pd.DataFrame(typology_recall).T.sort_values("recall")
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(typ))))
    ax.barh(typ.index, typ["recall"], color="#937860")
    ax.set_xlabel("recall @ 1% budget")
    ax.set_title("DNN -- recall by laundering typology")
    for i, (_, row) in enumerate(typ.iterrows()):
        ax.text(row["recall"] + 0.01, i, f"{int(row['caught'])}/{int(row['positives'])}",
                va="center")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "dnn_typology_recall.png")
    plt.show()
else:
    print("No typology labels in the test partition -- skipping.")


## 8. Save

Weights go out twice: a `state_dict` for convenience, and plain `.npz` arrays
for portability. This VM is big-endian, and the project's rule is that nothing
which has to survive a move between machines depends on a pickle -- `.npz`
carries its own dtype and byte order, a pickle carries the library version that
wrote it.


In [ ]:
torch.save(model.state_dict(), OUTPUT_DIR / "dnn_model.pt")
np.savez(OUTPUT_DIR / "dnn_weights.npz",
         **{k: v.numpy() for k, v in model.state_dict().items()})
np.save(OUTPUT_DIR / "dnn_scaler_mean.npy", scaler.mean_)
np.save(OUTPUT_DIR / "dnn_scaler_scale.npy", scaler.scale_)
np.save(OUTPUT_DIR / "dnn_test_scores.npy", dnn_scores)

dnn_results = {
    "model": "PyTorch MLP 128-64-32, dropout, isotonic-calibrated",
    "torch": torch.__version__, "machine": platform.machine(),
    "feature_source": GFP_FEATURE_SOURCE, "n_features": N_FEATURES,
    "epochs_run": len(history["loss"]), "best_epoch": best["epoch"] + 1,
    "pos_weight": float(pos_weight.item()),
    "train_seconds": round(train_seconds, 1),
    "val": dnn_val.to_metadata(), "test": dnn_test.to_metadata(),
    "typology_recall_at_1pct": typology_recall,
}
(OUTPUT_DIR / "dnn_results.json").write_text(
    json.dumps(dnn_results, indent=2, default=str), encoding="utf-8")
comparison.to_csv(OUTPUT_DIR / "dnn_comparison.csv")

print("wrote:")
for name in ("dnn_model.pt", "dnn_weights.npz", "dnn_scaler_mean.npy",
             "dnn_scaler_scale.npy", "dnn_test_scores.npy", "dnn_results.json",
             "dnn_comparison.csv"):
    print("  ", name)


## 9. Self-check

In [ ]:
assert not np.isnan(dnn_test.pr_auc), (
    "DNN PR-AUC is NaN -- almost always NaN features reaching the network. "
    "Check the nan_to_num step in batches()."
)
assert dnn_test.pr_auc > y_test.mean(), (
    f"DNN PR-AUC {dnn_test.pr_auc:.5f} is at or below the base rate "
    f"{y_test.mean():.5f} -- the model learned nothing."
)
assert len(dnn_scores) == len(y_test), "score/label length mismatch"
assert np.isfinite(dnn_scores).all(), "non-finite scores produced"

print("All self-checks passed.\n")
print(comparison.round(4).to_string())
